# Tennis AI — Colab 실험 준비

Python·패키지·GPU 환경을 확인하고 임시 실험 폴더와 환경 보고서를 만듭니다. **영상 분석·모델 학습·3D 생성은 아직 수행하지 않습니다.**

이 준비 단계는 CPU로 실행할 수 있습니다. 모델을 정한 뒤 필요할 때 Colab의 런타임 설정에서 GPU를 선택하세요. GPU 가용성과 실행 시간은 보장되지 않습니다.

[실행 안내](https://github.com/jwahyunkim/tennis-ai/blob/feature/video-platform/docs/colab.md) · [Colab 제한](https://research.google.com/colaboratory/faq.html)

셀은 위에서부터 실행하세요. 패키지 설치, 영상 업로드, Drive 연결, 모델 다운로드는 자동 실행하지 않습니다.

In [ ]:
import json
import platform
import shutil
import subprocess
import tempfile
from datetime import datetime, timezone
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

package_versions = {}
for package in (
    "torch",
    "tensorflow",
    "opencv-python",
    "opencv-python-headless",
    "onnxruntime",
    "onnxruntime-gpu",
    "numpy",
):
    try:
        package_versions[package] = version(package)
    except PackageNotFoundError:
        package_versions[package] = None

environment = {
    "checked_at_utc": datetime.now(timezone.utc).isoformat(),
    "python": platform.python_version(),
    "system": platform.system(),
    "packages": package_versions,
}
print(json.dumps(environment, indent=2, ensure_ascii=False))
print("null은 미설치 상태입니다. 모델 선정 후 필요한 패키지만 설치합니다.")

## GPU 확인

NVIDIA 장치와 드라이버가 보이는지만 확인합니다. 모델이 GPU에서 실행되는지, 메모리가 충분한지는 모델별 추론으로 따로 검증해야 합니다. CPU 환경에서도 다음 셀로 진행할 수 있습니다.

In [ ]:
gpu = {"status": "not_detected", "devices_csv": None}
nvidia_smi = shutil.which("nvidia-smi")
if nvidia_smi:
    try:
        result = subprocess.run(
            [
                nvidia_smi,
                "--query-gpu=name,memory.total,driver_version",
                "--format=csv,noheader",
            ],
            capture_output=True,
            text=True,
            timeout=10,
            check=False,
        )
        if result.returncode == 0 and result.stdout.strip():
            gpu = {"status": "detected", "devices_csv": result.stdout.strip()}
        else:
            gpu["status"] = "query_failed"
    except (OSError, subprocess.TimeoutExpired):
        gpu["status"] = "query_failed"
environment["gpu"] = gpu
print(json.dumps(gpu, indent=2, ensure_ascii=False))

## 임시 실험 폴더와 환경 보고서

Colab에서는 `/content`, 일반 로컬 실행에서는 시스템 임시 폴더 아래에 매번 새 폴더를 만듭니다. `inputs/`는 동의된 샘플, `models/`는 외부 가중치, `outputs/`는 결과용입니다. 이 셀은 빈 폴더와 환경 JSON만 생성합니다.

런타임이 사라지면 파일도 잃을 수 있습니다. 필요한 결과는 종료 전에 파일 패널에서 다운로드해 보관하세요. 영상·가중치·개인 결과는 Git에 넣지 않습니다.

In [ ]:
workspace_parent = "/content" if Path("/content").is_dir() else None
workspace = Path(tempfile.mkdtemp(prefix="tennis-ai-", dir=workspace_parent))
for folder in ("inputs", "models", "outputs"):
    (workspace / folder).mkdir()
report_path = workspace / "outputs" / "environment.json"
report_path.write_text(
    json.dumps(environment, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)
print(f"실험 폴더: {workspace}")
print(f"환경 보고서: {report_path}")

## 다음 실험 전에 정할 내용

1. T-02: 첫 동작·촬영 조건·원하는 2D/3D 결과·성공 기준. 기존 3D 목표는 유지합니다.
2. T-03: 샘플 영상 사용 권리·보관/삭제·학습 재사용 여부. 업로드 동의가 학습 동의를 뜻하지 않습니다.
3. T-04: 후보 모델·가중치·데이터 라이선스 확인, 의존성 버전과 모델 출처/버전 고정.

이후 사전학습 모델로 샘플을 추론하고 관절 추적 실패·처리시간·메모리를 기록합니다. 실행 코드는 재사용 가능한 Python 모듈로 분리하고 노트북에서는 실험만 조합합니다.

Git에 노트북을 저장하기 전 출력·실행 횟수·첨부 이미지를 비우고 토큰·개인 영상 경로가 없는지 확인하세요. `.gitignore`는 노트북 내부 출력을 제거하지 않습니다.